### i need to get date range for API parameters, so for that i am going to make a function that will deal with it


In [ ]:
from datetime import datetime as dt
from datetime import timedelta as td

import openmeteo_requests

import pandas as pd

import requests_cache
from retry_requests import retry

end_date = dt.now().strftime("%Y-%m-%d")
end_date

In [ ]:
start_date = dt.now() - td(days=7)
start_date = start_date.strftime("%y-%m-%d")
start_date

### this reminds me of the operator overloading i learned, notice we are subtracting class from class object.

### its working because in module we can define `__sub__` and control its behavior which allows it to handle such things


In [ ]:
def parameter_builder(file_path):
    df = pd.read_csv(file_path)

    # calcualting date based on current date
    end_date = dt.now().strftime("%Y-%m-%d")
    diff = dt.now() - td(days=7)
    start_date = diff.strftime("%Y-%m-%d")

    for _, row in df.iterrows():
        params = {
            "latitude": row["latitude"],
            "longitude": row["longitude"],
            "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
            "timezone": "auto",
            "start_date": start_date,
            "end_date": end_date,
        }

        yield row["site_code"], params

In [ ]:
# lets see if it works as intended
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    print(i, param)

### ok now i am gonna need a function that will fetch the data


In [ ]:
from datetime import datetime as dt
from datetime import timedelta as td
from retry_requests import retry
from sqlalchemy import engine

import openmeteo_requests
import pandas as pd
import requests_cache

# retries and backoff factors can handle errors
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

url = "https://api.open-meteo.com/v1/forecast"


def fetch_weather_data(site_code, params):
    responses = openmeteo.weather_api(url, params=params)
    response = responses[0]
    print(type(responses))
    print(response)

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)

### Testing the output we get


In [ ]:
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    fetch_weather_data(site_code=param[0], params=param[1])

In [ ]:
result = parameter_builder("meta_data.csv")
for i, param in enumerate(result):
    if i > 0:
        break
    print(param)

### My project Bottle Necks:

1. I am sending 10,000 requests one at a time which is stupid instead i can create a batch of 1000 sites which is a limit and get 1000 sites data in one request.

2. I am also writing the data in database frequently which is also stupid.

3. I should use `itertuple()` instead of `iterrows()`


### Why use intertuple() instead of interrows()?

iterrows() create pandas series which consumes time instead intertuple is like python generator, it creates NameTuples like:

```
Pandas(
    index=1,
    site_code=XER10,
    latitude = 37.45,
    longitude = 70.34
)
```

in other words they are like:

```
yield(
   index=1,
    site_code=XER10,
    latitude = 37.45,
    longitude = 70.34
)
```


### why DataFrame() is faster?

- its because DataFrame() in pandas create each colum into numpy.array() which is very beneficial as numpy uses C language for execution


### I am going to try to get all the data in batches of 100 sites per request


## Method 1:


In [ ]:
from datetime import datetime as dt
from datetime import timedelta as td
from retry_requests import retry

import openmeteo_requests
import pandas as pd
import requests_cache
import database as db
import time

# cache_session, retries and backoff factors
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=2, backoff_factor=0.5)
openmeteo = openmeteo_requests.Client(session=retry_session)

# lets define constant varaibles
URL = "https://api.open-meteo.com/v1/forecast"
PATH = "meta_data.csv"


# just a class for raising custom built error
class APIRateLimitError(Exception):
    "Raised when the Open-Meteo hourly rate limit exceeded"

    pass


def create_meta_table():
    db.create_sites_table()


def create_weather_data():
    db.create_weather_table()


def insert_meta_data(path):
    df = pd.read_csv(path)
    db.insert_meta_data(df)


def insert_weather_data(df):
    db.insert_weather_data(df)


def batch_builder(file_path):
    df = pd.read_csv(file_path)

    # batch_size for each request 100 is the limit
    batch_size = 100

    for start_index in range(0, len(df), batch_size):

        # now i have a chunk of dataframe that i can work with
        df_batch = df.iloc[start_index : start_index + batch_size]

        yield df_batch


def data_parser(site_code, response):
    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)

    return hourly_dataframe


def fetch_weather_data(df_batch):
    attempts = 3

    # calcualting date based on current date
    end_date = (dt.now() + td(days=1)).strftime("%Y-%m-%d")
    diff = (dt.now() + td(days=1)) - td(days=8)
    start_date = diff.strftime("%Y-%m-%d")

    # parameters for request
    params = {
        "latitude": df_batch["latitude"].tolist(),
        "longitude": df_batch["longitude"].tolist(),
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "shortwave_radiation",
        ],
        "timezone": "auto",
        "start_date": start_date,
        "end_date": end_date,
    }
    for attempt in range(attempts):
        try:
            responses = openmeteo.weather_api(url=URL, params=params)
            break

        except Exception as e:
            error_message = str(e)
            # print(f"{site_code} failed")
            print(f"Attempt {attempt + 1} / {attempts}")
            print(f"reason of faliure {e}")

            # raise error if hourly limit reached
            if "Hourly API request limit exceeded" in error_message:
                raise APIRateLimitError("API hourly limit reached")

            # if request failed then retry that request after 5 sec
            if attempt < attempts - 1:
                print("retrying after 5 seconds \n")
                time.sleep(5)

            else:
                print(f"request still failed after {attempts} attempts")
                print(f"reason of faliure: {e}")
                return None

        weather_data_batch = []
        for site_code, response in zip(df_batch["site_code"], responses):
            weather_data_batch.append(data_parser(site_code, response))

        return pd.concat(weather_data_batch, ignore_index=True)


def run_weather_etl(path):

    insert_meta_data(path)
    count = 0

    try:
        for df_batch in batch_builder(path):
            count += 1
            df = fetch_weather_data(df_batch)

            if df is not None:
                insert_weather_data(df)
                print(count)

    except APIRateLimitError as e:
        print(e)
        print("stopping ETL because rate limit reached")

### Writing the code again with a different approach

- we are gona use .apply() instead of generator and for loop see if it makes it faster
- The response object from openmeteo_requests isn't a plain Python dict/JSON — it's a special object from their SDK (backed by FlatBuffers).


### How requests_cache + SQLite works:

- It stores a mapping: cache key → full HTTP response (headers, body, status code — everything).
- The cache key is built from the request itself: method + URL + all query parameters (your params dict), hashed together.
- When you make a request, it checks: "have I seen this exact URL+params combination before, within expire_after seconds?" If yes → returns the stored response, no network call. If no → sends the real request, then stores the new response under a new key.


### When openmeteo_requests.Client.weather_api() runs, under the hood it does two things in sequence:

- Sends a plain HTTP GET request (via the session you gave it) → gets back raw bytes in the response body (FlatBuffers binary format, not JSON).
- Immediately parses those bytes into a WeatherApiResponse object using a function called WeatherApiResponse.GetRootAs() (from the openmeteo_sdk package) — this is what gives you .Latitude(), .Hourly(), etc.


### Problems with the following design:

- .apply() restarts everytime we can not create a variable that can store something and when .apply reads another row it will reset. So we have to store that variable outside the function that we will use in .apply().

- If parser() catches its own exception internally (like i am doing with try/except) and just returns normally — .apply() keeps calling parser() for the next row automatically. No crash, no restart. This part is fine.

- If an exception escapes parser() uncaught, the entire .apply() call dies immediately — and yes, that matches what my instructor said: "you'd have to rerun the whole script from row 1, and any progress not yet saved elsewhere is lost."

- .apply() either finishes completely or dies with nothing returned


## Method 2:


In [ ]:
from datetime import datetime as dt
from datetime import timedelta as td
from retry_requests import retry

import openmeteo_requests
import pandas as pd
import requests_cache
import database as db
import time
import sys
import csv

# retries and backoff factors
retry_session = retry(retries=2, backoff_factor=0.5)
openmeteo = openmeteo_requests.Client(session=retry_session)

# lets define constant varaibles
URL = "https://api.open-meteo.com/v1/forecast"
PATH = "meta_data.csv"
START_TIME = dt.now()
TRACKING_PATH = "tracking.csv"


def create_meta_table():
    db.create_sites_table()


def create_weather_data():
    db.create_weather_table()


def insert_meta_data(path):
    df = pd.read_csv(path)
    db.insert_meta_data(df)


def insert_weather_data(df):
    db.insert_weather_data(df)


def log_status(site_code, status):
    """
    This function will append site_code and and status into a tracking csv file,
    which will keep the track of which site's data extraction successed and which failed
    """
    with open(TRACKING_PATH, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([site_code, status])


def parser(row):
    site_code = row["site_code"]

    # calcualting date based on current date
    end_date = (dt.now() + td(days=1)).strftime("%Y-%m-%d")
    diff = (dt.now() + td(days=1)) - td(days=8)
    start_date = diff.strftime("%Y-%m-%d")

    # parameters for request
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
        "timezone": "auto",
        "start_date": start_date,
        "end_date": end_date,
    }

    try:
        # sending request to API
        responses = openmeteo.weather_api(url=URL, params=params)

    except Exception as e:
        error_message = str(e)

        # raise error if hourly limit reached
        if "Hourly API request limit exceeded" in error_message:
            time_elapsed = dt.now() - START_TIME
            remaining_time = td(hours=1) - time_elapsed
            wait_seconds = max(
                remaining_time.total_seconds(), 0
            )  # in case if Hourly limit error hit unexpectidly after an hour for some reason
            time.sleep(wait_seconds)
            print(f"Hourly API request limit exceeded: {wait_seconds}")

        elif "Minutely API request limit exceeded" in error_message:
            time_elapsed = dt.now() - START_TIME
            remaining_time = td(minutes=1) - time_elapsed
            wait_seconds = max(
                remaining_time.total_seconds(), 0
            )  # same reason as above
            time.sleep(wait_seconds)
            print(f"Minutely API request limit exceeded, waiting for: {wait_seconds}")

        elif "Daily API request limit exceeded" in error_message:
            sys.exit("Daily API request limit reached, give it a rest see ya tomorrow")

        # as all conditions are exausted we need to update the status as a faliure
        # and i am returning None so i can handle return values in fetch_weather_data()
        log_status(site_code=site_code, status=False)
        return None

    response = responses[0]

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)

    # response did not failed so status is going to be True
    log_status(site_code=site_code, status=True)
    return hourly_dataframe


def new_sites_fetch_data(path):
    # changing the global variable here in case if we import in function in another file
    # and call the function there it will not reset the start time after the first execution.
    global START_TIME
    START_TIME = dt.now()

    # inserting the meta data in database
    insert_meta_data(path)

    # refreshing the tracking.csv first
    with open("tracking.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["site_code", "status"])

    # fetching the data
    df = pd.read_csv(path)
    df_series = df.apply(parser, axis=1)

    # filtered the data as failed values returned None
    df_filtered = [data for data in df_series if data is not None]

    # a small check in case if the list is empty
    if not df_filtered:
        print("There is nothing to insert into database, Please try again!")
        return

    result_df = pd.concat(df_filtered)

    # inserting the data in database
    insert_weather_data(result_df)


def failed_sites_retry(path):
    # changing the global variable here in case if we import in function in another file
    # and call the function there it will not reset the start time after the first execution.
    global START_TIME
    START_TIME = dt.now()

    # using tracker to filter sites that failed
    df = pd.read_csv(path)
    tracker = pd.read_csv("tracking.csv")

    # filtering only sites that failed
    failed_sites = tracker.loc[tracker["status"] == False]
    mask = df["site_code"].isin(failed_sites["site_code"])
    df_failed_sites = df.loc[mask]

    # now using dataframe of failed sites only we are going to retry them
    df_series = df_failed_sites.apply(parser, axis=1)
    df_filtered = [data for data in df_series if data is not None]

    # a small check in case if the list is empty
    if not df_filtered:
        print("There is nothing to insert into database, Please try again!")
        return
    result_df = pd.concat(df_filtered)

    # now we are going to insert the filtered data in database
    insert_weather_data(result_df)

## Improvements:

- Any site must not fail when we try for failed sites, we are gonna use recurrsion or while loop for that.


In [2]:
from datetime import datetime as dt
from datetime import timedelta as td

import openmeteo_requests
import pandas as pd
import database as db
import time
import sys

openmeteo = openmeteo_requests.Client()

# lets define constant varaibles
URL = "https://api.open-meteo.com/v1/forecast"
PATH = "meta_data.csv"


def create_meta_table():
    db.create_sites_table()


def create_weather_data():
    db.create_weather_table()


def create_tracker_table():
    db.create_tracking_table()


def insert_meta_data(path):
    df = pd.read_csv(path)
    db.insert_meta_data(df)


def insert_weather_data(df):
    db.insert_weather_data(df)


def parser(row):
    site_code = row["site_code"]

    # calcualting date based on current date
    end_date = (dt.now() + td(days=1)).strftime("%Y-%m-%d")
    diff = (dt.now() + td(days=1)) - td(days=8)
    start_date = diff.strftime("%Y-%m-%d")

    # parameters for request
    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
        "timezone": "auto",
        "start_date": start_date,
        "end_date": end_date,
    }

    while True:
        try:
            responses = openmeteo.weather_api(url=URL, params=params)
            break

        except Exception as e:
            error_message = str(e)

            print("EXCEPTION TYPE:", type(e))
            print("EXCEPTION:", repr(e))

            if "Hourly API request limit exceeded" in error_message:
                print("Hourly API request limit exceeded. Waiting 1 hour...")
                time.sleep(3600)
                continue

            elif "Minutely API request limit exceeded" in error_message:
                print("Minutely API request limit exceeded. Waiting 1 minute...")
                time.sleep(60)
                continue

            elif "Daily API request limit exceeded" in error_message:
                sys.exit(
                    "Daily API request limit reached, give it a rest see ya tomorrow"
                )

            db.update_tracking_status(site_code=site_code, status=False)
            return None

    response = responses[0]

    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_global_tilted_irradiance_instant = hourly.Variables(2).ValuesAsNumpy()

    hourly_data = {
        "date": pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )
    }

    hourly_data["site_code"] = site_code
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["global_tilted_irradiance_instant"] = (
        hourly_global_tilted_irradiance_instant
    )

    hourly_dataframe = pd.DataFrame(data=hourly_data)

    # response did not failed so status is going to be True
    db.update_tracking_status(site_code=site_code, status=True)
    return hourly_dataframe


def new_sites_fetch_data(path):
    # changing the global variable here in case if we import in function in another file
    # and call the function there it will not reset the start time after the first execution.

    # batch size
    batch_size = 100

    # inserting the meta data in database
    insert_meta_data(path)

    # fetching the data and filtering the sites that are already done
    df = pd.read_csv(path)
    tracker = db.read_parsed_sites()

    mask = ~df["site_code"].isin(tracker["site_code"])
    df = df.loc[mask]

    # instead of getting the data of 10,000 sites we are gonna insert 100 sites data over time
    for start in range(0, len(df), batch_size):

        batch_df = df.iloc[start : start + batch_size]
        df_series = batch_df.apply(parser, axis=1)

        # filtered the data as failed values returned None
        parsed_data = [data for data in df_series if data is not None]

        # a small check in case if the list is empty
        if not parsed_data:
            print("There is nothing to insert into database, Please try again!")
            return

        result_df = pd.concat(parsed_data)

        # inserting the data in database
        insert_weather_data(result_df)


def failed_sites_retry(path):
    # changing the global variable here in case if we import in function in another file
    # and call the function there it will not reset the start time after the first execution.
    attempts = 0
    while True:

        # using tracker to filter sites that failed
        df = pd.read_csv(path)
        failed_sites = db.read_failed_sites()

        # if there are no failed site break the loop
        if failed_sites.empty:
            break

        mask = df["site_code"].isin(failed_sites["site_code"])
        df_failed_sites = df.loc[mask]

        # now using dataframe of failed sites only we are going to retry them
        df_series = df_failed_sites.apply(parser, axis=1)

        df_filtered = [data for data in df_series if data is not None]

        if not df_filtered:
            attempts += 1
            print(f"Still no data is returned, attempt Number: {attempts}")
            continue

        result_df = pd.concat(df_filtered)

        # now we are going to insert the filtered data in database
        insert_weather_data(result_df)

In [ ]:
create_meta_table()
create_weather_data()
create_tracker_table()


new_sites_fetch_data(PATH)

In [1]:
import requests
import json
import pandas as pd

params = {
    "latitude": 42.3601,
    "longitude": -71.0589,
    "hourly": ["temperature_2m", "relative_humidity_2m", "shortwave_radiation"],
    "timezone": "auto",
}

response = requests.get("https://api.open-meteo.com/v1/forecast", params=params)

data = response.json()

data["hourly"]["time"] = (
    pd.to_datetime(
        data["hourly"]["time"],
        utc=True,
    )
    .strftime("%Y-%m-%d %H:%M:%S%z")
    .tolist()
)

print(data["hourly"])

{'time': ['2026-09-22 00:00:00+0000', '2026-09-22 01:00:00+0000', '2026-09-22 02:00:00+0000', '2026-09-22 03:00:00+0000', '2026-09-22 04:00:00+0000', '2026-09-22 05:00:00+0000', '2026-09-22 06:00:00+0000', '2026-09-22 07:00:00+0000', '2026-09-22 08:00:00+0000', '2026-09-22 09:00:00+0000', '2026-09-22 10:00:00+0000', '2026-09-22 11:00:00+0000', '2026-09-22 12:00:00+0000', '2026-09-22 13:00:00+0000', '2026-09-22 14:00:00+0000', '2026-09-22 15:00:00+0000', '2026-09-22 16:00:00+0000', '2026-09-22 17:00:00+0000', '2026-09-22 18:00:00+0000', '2026-09-22 19:00:00+0000', '2026-09-22 20:00:00+0000', '2026-09-22 21:00:00+0000', '2026-09-22 22:00:00+0000', '2026-09-22 23:00:00+0000', '2026-09-23 00:00:00+0000', '2026-09-23 01:00:00+0000', '2026-09-23 02:00:00+0000', '2026-09-23 03:00:00+0000', '2026-09-23 04:00:00+0000', '2026-09-23 05:00:00+0000', '2026-09-23 06:00:00+0000', '2026-09-23 07:00:00+0000', '2026-09-23 08:00:00+0000', '2026-09-23 09:00:00+0000', '2026-09-23 10:00:00+0000', '2026-09-2

In [7]:
from data_pipeline import fetch_site_weather

result_json = fetch_site_weather("site_code", 42.3601, -71.0589)
result_json["date"] = result_json.pop("time")

print(result_json.keys())

dict_keys(['temperature_2m', 'relative_humidity_2m', 'shortwave_radiation', 'site_code', 'date'])
